# Día 6 · Unidad 6 — Código profesional (II): excepciones

**Objetivos de aprendizaje**
- Distinguir errores de sintaxis de excepciones, y entender por qué un programa se detiene al encontrar una.
- Usar `raise` y `assert` para señalar condiciones inválidas, y saber cuándo usar cada uno.
- Gestionar excepciones con `try`/`except`/`else`/`finally`, capturando el tipo de error correcto (nunca un `except:` desnudo).
- Crear excepciones propias heredando de `Exception` para dar mensajes de error específicos del dominio.

**Duración estimada:** 75-90 minutos.

**Requisitos previos:** Día 4 (funciones), Día 5 (clases). Este notebook depende del archivo `ejemplo.txt` de esta misma carpeta.

## 1. Motivación: errores de sintaxis vs. excepciones

Un programa de Python se detiene en cuanto encuentra un error. Hay dos tipos:

- **Error de sintaxis**: el código ni siquiera es Python válido. Python lo detecta antes de ejecutar nada.
- **Excepción**: el código es sintácticamente correcto, pero algo falla durante la ejecución (dividir por cero, abrir un archivo que no existe, sumar un texto con un número...).

Entender y gestionar excepciones es lo que separa un script que "funciona en el caso feliz" de código robusto que se puede poner en producción.

In [ ]:
# Error de sintaxis: falta un paréntesis de cierre. Python ni siquiera llega a ejecutar la línea.
# (celda comentada a propósito: si se ejecutara sin comentar, el notebook no podría continuar)
# print(0/0

# Ahora sí es sintácticamente correcto, pero es una EXCEPCIÓN en tiempo de ejecución:
try:
    print(0 / 0)
except ZeroDivisionError as e:
    print(f"Excepción capturada: {type(e).__name__}: {e}")

Python indica siempre **qué tipo** de excepción se ha producido (`ZeroDivisionError` en este caso). Hay muchas excepciones ya integradas (`ValueError`, `TypeError`, `KeyError`, `FileNotFoundError`...) y también se pueden crear propias (sección 5).

## 2. La cláusula `raise`

`raise` levanta una excepción explícitamente cuando se detecta una condición inválida. Es la forma correcta de decir "esta situación no es válida y el resto del programa no debe continuar como si nada".

In [ ]:
from dataclasses import dataclass


@dataclass
class Poliza:
    asegurado: str
    edad: int


def suscribir_poliza(asegurado: str, edad: int) -> Poliza:
    """Da de alta una póliza si la edad del asegurado es válida."""
    if edad <= 0:
        raise ValueError(f"Edad de asegurado inválida: {edad}. Debe ser > 0")
    return Poliza(asegurado=asegurado, edad=edad)


suscribir_poliza(asegurado="Anna Puig", edad=34)

Si la edad no es válida, `raise` detiene la ejecución con un mensaje explícito — mucho más útil para quien tenga que depurarlo que un fallo silencioso o un resultado incorrecto:

In [ ]:
try:
    suscribir_poliza(asegurado="Nen Petit", edad=-3)
except ValueError as e:
    print(f"No se ha podido suscribir la póliza: {e}")

## 3. `assert`

`assert` comprueba que una condición es verdadera; si es `False`, levanta `AssertionError` con el mensaje que le indiques. Se usa sobre todo para comprobar **invariantes internos** durante el desarrollo — condiciones que "nunca deberían fallar si el resto del código es correcto" — no para validar datos que vienen de fuera (un usuario, una API, un archivo).

In [ ]:
def calcular_prima_media(primas: list[float]) -> float:
    assert len(primas) > 0, "La lista de primas no puede estar vacía"
    return sum(primas) / len(primas)


print(calcular_prima_media([420.0, 260.0, 500.0]))

try:
    calcular_prima_media([])
except AssertionError as e:
    print(f"AssertionError: {e}")

> ⚠️ **Error típico**: usar `assert` para validar entradas de usuario en código de producción. Los `assert` se **eliminan por completo** si Python se ejecuta con el flag de optimización `-O` (`python -O mi_script.py`) — cualquier validación que dependa de un `assert` desaparece silenciosamente. Para validar datos que vienen de fuera (un formulario, una API, un archivo de un tercero), usa siempre `raise` con una excepción explícita, como en la sección anterior.

## 4. Gestión de excepciones: `try` / `except`

La forma más básica:

```python
try:
    <código que puede fallar>
except <TipoDeError>:
    <código que se ejecuta si falla>
```

En días anteriores has visto celdas con `input()` para simular interacción del usuario; en este notebook, para que se pueda ejecutar de principio a fin sin bloquearse esperando teclado, la "entrada del usuario" es una variable ya asignada — el patrón es idéntico al de un `input()` real, solo cambia de dónde viene el valor.

In [ ]:
# Simulamos la entrada de un formulario de suscripción (en vez de bloquear con input())
edad_introducida = "no-es-un-numero"  # prueba a cambiar este valor y reejecutar la celda

try:
    edad_asegurado = int(edad_introducida)
    poliza = suscribir_poliza(asegurado="Marc Soler", edad=edad_asegurado)
    print(poliza)
except ValueError as e:
    print(f"Dato inválido, no se ha podido suscribir la póliza: {e}")

Fíjate en que este `except ValueError` captura **dos** fallos distintos con la misma línea: que `int(...)` no pueda convertir el texto (`ValueError: invalid literal for int()`) y que `suscribir_poliza` rechace una edad no válida (nuestro propio `raise ValueError`). Ambos son, de hecho, la misma clase de excepción.

> ⚠️ **Error típico**: usar `except:` a secas (sin especificar el tipo). Captura *cualquier* excepción, incluidas las que no esperabas — por ejemplo, un error de programación real (`NameError` por una variable mal escrita) quedaría silenciado igual que un dato inválido, y sería mucho más difícil de detectar.

In [ ]:
# Mal: captura todo, incluidos errores de programación que deberían hacer saltar una alarma
def demo_except_desnudo():
    try:
        resultado = suscribir_poiza(asegurado="X", edad=30)  # nombre de función mal escrito a propósito
    except:
        print("Algo ha fallado")  # ¿qué ha fallado? no lo sabemos


demo_except_desnudo()

Si especificamos el tipo correcto, ese error de programación **no** queda oculto — y eso es justo lo que queremos:

In [ ]:
def demo_except_especifico():
    try:
        resultado = suscribir_poiza(asegurado="X", edad=30)  # sigue mal escrito a propósito
    except ValueError as e:
        print(f"Dato inválido: {e}")
    # NameError NO se captura aquí: se propaga y se ve claramente que hay un bug real


try:
    demo_except_especifico()
except NameError as e:
    print(f"Esto es un bug de programación, no un dato inválido: {type(e).__name__}: {e}")

Se pueden capturar varios tipos a la vez, bien con varias cláusulas `except`, bien agrupándolos en una tupla:

```python
except (ValueError, TypeError):
    ...
```

## 5. Cláusula `else`

`else` ejecuta un bloque solo si el `try` **no** lanzó ninguna excepción. Es útil para separar claramente "el código que puede fallar" del "código que depende de que haya funcionado":

In [ ]:
for edad_introducida in ["34", "-5", "no-numerico"]:
    try:
        edad_asegurado = int(edad_introducida)
        poliza = suscribir_poliza(asegurado="Anna Puig", edad=edad_asegurado)
    except ValueError as e:
        print(f"Rechazado ({edad_introducida!r}): {e}")
    else:
        print(f"Aceptado: {poliza}")

## 6. Cláusula `finally`

`finally` se ejecuta **siempre**, haya habido excepción o no — típicamente para liberar un recurso (cerrar un archivo, una conexión de base de datos). Vamos a leer el archivo `ejemplo.txt` de esta carpeta:

In [ ]:
archivo = None
try:
    archivo = open("ejemplo.txt", "r", encoding="utf-8")
    contenido = archivo.read()
except FileNotFoundError:
    print("Archivo no encontrado")
    contenido = ""
else:
    print("Archivo leído correctamente")
    contenido = contenido.upper()
finally:
    if archivo is not None:
        archivo.close()
    print(f"¿Archivo cerrado?: {archivo.closed if archivo else 'no llegó a abrirse'}")

print(f"Contenido: {contenido!r}")

Si el archivo no existe, `finally` se ejecuta igualmente, pero sin intentar cerrar nada que no llegó a abrirse (por eso el `if archivo is not None`):

In [ ]:
archivo = None
try:
    archivo = open("ejemplo_que_no_existe.txt", "r", encoding="utf-8")
    contenido = archivo.read()
except FileNotFoundError:
    print("Archivo no encontrado")
    contenido = ""
finally:
    if archivo is not None:
        archivo.close()
    print("Bloque finally ejecutado de todas formas")

> 📊 En Excel esto sería como usar `=SI.ERROR(fórmula; valor_si_error)`: en vez de que un `#DIV/0!` rompa toda la hoja y arrastre el error a cada celda que dependa de ella, decides explícitamente qué valor devolver si la fórmula falla. `try/except` es lo mismo, pero con control total sobre qué tipo de error capturas y qué hacer en cada caso — no un único valor de repuesto para cualquier fallo.

En el ejemplo de arriba, `with open(...) as archivo:` habría sido más idiomático que gestionar el cierre a mano en `finally` (Python cierra el archivo automáticamente al salir del bloque `with`, incluso si hay excepción). Se ha mostrado la versión con `finally` porque el objetivo aquí es entender qué hace cada cláusula — en tu propio código, para archivos usa siempre `with`.

In [ ]:
with open("ejemplo.txt", "r", encoding="utf-8") as archivo:
    contenido = archivo.read()

print(contenido.upper())
print("¿Cerrado tras salir del with?:", archivo.closed)

**Para ti:** escribe una función `leer_prima_desde_texto(texto: str) -> float` que intente convertir `texto` a `float` y devuelva el valor. Si falla, debe capturar el error e imprimir un mensaje indicando qué texto no se pudo convertir, y devolver `0.0`. Pruébala con `"450.5"` y con `"cuatrocientos"`.

In [ ]:
# Tu código aquí


## 7. Construir excepciones propias

Las excepciones integradas (`ValueError`, `TypeError`...) son genéricas. En un proyecto real, suele merecer la pena crear excepciones propias que documenten el error de negocio con su propio nombre — heredando de `Exception` (o de una excepción más específica, como `ValueError`):

In [ ]:
class PolizaInvalidaError(ValueError):
    """Se lanza cuando los datos de una póliza no cumplen las reglas de suscripción."""


class EdadNoAseguradaError(PolizaInvalidaError):
    """Caso específico: la edad del asegurado está fuera del rango asegurable."""


def suscribir_poliza_v2(asegurado: str, edad: int, prima: float) -> Poliza:
    """Versión con excepciones propias, más específicas que un ValueError genérico."""
    if not (18 <= edad <= 75):
        raise EdadNoAseguradaError(
            f"Edad {edad} fuera de rango asegurable (18-75) para {asegurado}"
        )
    if prima <= 0:
        raise PolizaInvalidaError(f"Prima inválida ({prima}) para {asegurado}")
    return Poliza(asegurado=asegurado, edad=edad)


for edad_prueba in [10, 34, 90]:
    try:
        poliza = suscribir_poliza_v2(asegurado="Test", edad=edad_prueba, prima=300.0)
        print(f"OK: {poliza}")
    except EdadNoAseguradaError as e:
        print(f"Rechazado por edad: {e}")
    except PolizaInvalidaError as e:
        print(f"Rechazado (otro motivo): {e}")

`EdadNoAseguradaError` hereda de `PolizaInvalidaError`, que a su vez hereda de `ValueError`. Esto significa que se puede capturar con cualquiera de los tres tipos, de más específico a más genérico — útil para que el código que llama decida el nivel de detalle con el que quiere reaccionar. Es el mismo principio de jerarquía que ya usaste con la herencia de clases en el Día 5.

**Para ti:** crea una excepción `RecargoInvalidoError(ValueError)` y una función `aplicar_recargo(prima: float, recargo: float) -> float` que la levante si `recargo` es negativo o mayor que `1` (100%). Compruébala con `recargo=0.15` (debe funcionar) y `recargo=2.5` (debe lanzar la excepción).

In [ ]:
# Tu código aquí


## Resumen

- Un error de sintaxis impide que el código llegue a ejecutarse; una excepción ocurre durante la ejecución de código sintácticamente válido.
- `raise` levanta una excepción explícitamente ante una condición inválida — es la herramienta correcta para validar datos externos.
- `assert` comprueba invariantes internos durante el desarrollo; se descarta con `python -O`, así que nunca debe ser la única validación de datos que vienen de fuera.
- `try`/`except` captura excepciones; especifica siempre el tipo (`except ValueError`), nunca un `except:` desnudo — de lo contrario ocultas también bugs reales de programación.
- `else` ejecuta código solo si el `try` no falló; `finally` se ejecuta siempre, se use o no para liberar recursos (aunque para archivos, `with` es más idiomático que un `finally` manual).
- Crear excepciones propias (heredando de `Exception` o de una excepción más específica) documenta errores de negocio con su propio nombre y permite capturarlos con distintos niveles de granularidad.

**Siguiente:** `03_teoria_logging.ipynb`, donde veremos cómo registrar lo que ocurre durante la ejecución con el módulo estándar `logging`.